# Задание 1 — Формирование датасета

**Сценарий:** два автомобиля подъезжают к перекрёстку. Нужно определить, произойдёт ли столкновение (ДТП).

**Признаки каждого автомобиля:**

| Признак | Тип | Описание |
|---|---|---|
| `is_moving` | бинарный | едет ли автомобиль (70% = да) |
| `car_type` | номинальный | sedan / suv / truck / motorcycle |
| `speed_level` | порядковый | 0=стоит, 1=<30 км/ч, 2=30–60 км/ч, 3=>60 км/ч |
| `dist_to_intersection` | количественный | расстояние до перекрёстка (м) |
| `direction` | порядковый | 0=N, 1=E, 2=S, 3=W — с какой стороны едет |
| `has_priority` | бинарный | едет по главной дороге |
| `road_condition` | номинальный | dry / wet / icy / snowy (n≥8) |
| `visibility_level` | порядковый | 0=плохая, 1=средняя, 2=хорошая (n≥8) |
| `has_braked` | бинарный | водитель нажал тормоз (n≥8) |
| `reaction_time` | количественный | время реакции водителя, сек (n≥9) |
| `traffic_light` | порядковый | 0=красный, 1=жёлтый, 2=зелёный (n≥10) |
| `is_speeding` | бинарный | превышает скорость (n≥10) |
| `driver_attention` | порядковый | 0=отвлёкся, 1=норма, 2=сосредоточен (n≥11) |

**Целевой признак `collision`:** оба едут и разница во времени прибытия к перекрёстку меньше порога.  
Порог увеличивается если оба на главной дороге, уменьшается если кто-то тормозит.

In [6]:
import numpy as np
import pandas as pd

In [7]:
# Скорость в м/с для каждого уровня speed_level (0 — стоит, используется только если is_moving=0)
_SPEED_MS = {1: 8.0, 2: 14.0, 3: 20.0}

def generate_car(n_features, rng):
    """Генерирует признаки одного автомобиля."""
    car = {}

    # --- базовые признаки (n_features = 4..7) ---
    car['is_moving']            = int(rng.integers(0, 10) < 7)                               # бинарный (70% едет)
    car['car_type']             = rng.choice(['sedan', 'suv', 'truck', 'motorcycle'])        # номинальный
    # если едет — скорость 1..3, если стоит — 0
    car['speed_level']          = int(rng.integers(1, 4)) if car['is_moving'] else 0         # порядковый: 0..3
    car['dist_to_intersection'] = float(rng.uniform(2, 40))                                  # количественный (м)
    car['direction']            = int(rng.integers(0, 4))                                    # порядковый: 0=N,1=E,2=S,3=W
    car['has_priority']         = int(rng.integers(0, 2))                                    # бинарный

    if n_features >= 7:
        car['speed_kmh'] = float(rng.uniform(0, 90))                                         # количественный (км/ч)

    # --- n_features = 8..9 ---
    if n_features >= 8:
        car['road_condition']   = rng.choice(['dry', 'wet', 'icy', 'snowy'])                 # номинальный
        car['visibility_level'] = int(rng.integers(0, 3))                                    # порядковый: 0..2
        car['has_braked']       = int(rng.integers(0, 2))                                    # бинарный

    if n_features >= 9:
        car['reaction_time'] = float(rng.uniform(0.5, 2.5))                                  # количественный (сек)

    # --- n_features >= 10 ---
    if n_features >= 10:
        car['traffic_light'] = int(rng.integers(0, 3))                                       # порядковый: 0=красный..2=зелёный
        car['is_speeding']   = int(rng.integers(0, 2))                                       # бинарный

    if n_features >= 11:
        car['driver_attention'] = int(rng.integers(0, 3))                                    # порядковый: 0=отвлёкся..2=сосредоточен

    return car

In [8]:
def is_collision(car1, car2):
    """
    ДТП происходит если:
    1. Оба автомобиля едут (is_moving = 1)
    2. Разница во времени прибытия к перекрёстку < порога

    Время прибытия = dist_to_intersection / скорость
    Порог = 1.5 сек (базовый)
        +0.8 сек  если оба едут по главной (никто не уступает)
        -0.5 сек  если хотя бы один нажал тормоз
        +0.5 сек  если у кого-то плохая видимость (visibility_level=0)
    """
    if car1['is_moving'] == 0 or car2['is_moving'] == 0:
        return 0

    v1 = _SPEED_MS[car1['speed_level']]
    v2 = _SPEED_MS[car2['speed_level']]

    t1 = car1['dist_to_intersection'] / v1
    t2 = car2['dist_to_intersection'] / v2

    base = 1.5

    if car1['has_priority'] == 1 and car2['has_priority'] == 1:
        base += 0.8   # оба считают что у них приоритет — никто не уступает

    if car1.get('has_braked', 0) or car2.get('has_braked', 0):
        base -= 0.5   # кто-то успел затормозить

    if car1.get('visibility_level', 1) == 0 or car2.get('visibility_level', 1) == 0:
        base += 0.5   # плохая видимость — поздно заметили друг друга

    return int(abs(t1 - t2) < base)

In [9]:
def generate_dataset(n_samples, n_features, seed=42):
    """Генерирует датасет из n_samples строк с n_features признаками на автомобиль."""
    rng = np.random.default_rng(seed)
    rows = []
    for _ in range(n_samples):
        car1 = generate_car(n_features, rng)
        car2 = generate_car(n_features, rng)
        row = {f'car1_{k}': v for k, v in car1.items()}
        row.update({f'car2_{k}': v for k, v in car2.items()})
        row['collision'] = is_collision(car1, car2)
        rows.append(row)
    return pd.DataFrame(rows)

In [10]:
# 12 датасетов: 4 размера × 3 группы признаков
SIZES      = [60, 300, 750, 1500]  
N_FEATURES = [5, 9, 11]             

datasets = {}
for n_feat in N_FEATURES:
    for n_samples in SIZES:
        key = f'n{n_feat}_s{n_samples}'
        df = generate_dataset(n_samples, n_feat, seed=n_feat * 1000 + n_samples)
        datasets[key] = df
        df.to_csv(f'datasets/{key}.csv', index=False)
        print(f'{key}: {df.shape}, ДТП: {df["collision"].sum()} ({df["collision"].mean():.1%})')

n5_s60: (60, 13), ДТП: 20 (33.3%)
n5_s300: (300, 13), ДТП: 110 (36.7%)
n5_s750: (750, 13), ДТП: 282 (37.6%)
n5_s1500: (1500, 13), ДТП: 489 (32.6%)
n9_s60: (60, 23), ДТП: 16 (26.7%)
n9_s300: (300, 23), ДТП: 108 (36.0%)
n9_s750: (750, 23), ДТП: 249 (33.2%)
n9_s1500: (1500, 23), ДТП: 479 (31.9%)
n11_s60: (60, 29), ДТП: 21 (35.0%)
n11_s300: (300, 29), ДТП: 95 (31.7%)
n11_s750: (750, 29), ДТП: 228 (30.4%)
n11_s1500: (1500, 29), ДТП: 509 (33.9%)


In [11]:
# Пример датасета
datasets['n5_s60'].head()

,car1_is_moving,car1_car_type,car1_speed_level,car1_dist_to_intersection,car1_direction,car1_has_priority,car2_is_moving,car2_car_type,car2_speed_level,car2_dist_to_intersection,car2_direction,car2_has_priority,collision
0,1,suv,1,33.717229,1,1,0,suv,0,25.440733,0,1,0
1,1,suv,3,6.058430,3,0,1,sedan,3,32.743946,0,0,1
2,0,sedan,0,19.297831,0,1,0,motorcycle,0,17.025311,1,0,0
3,1,sedan,2,39.175837,1,1,1,suv,2,27.824838,3,0,1
4,1,truck,1,38.909653,0,0,1,sedan,3,34.155671,1,1,0
